# 11. 품목명(`train_prdlst_nm`)별 검출률

- 원자료: `listeria_body.parquet` 전체 행
- 전체 / 버섯 제외(노트북 9·10 과 같은 `prduct_nm` 키워드 기준) 를 나란히 본다
- 검출비중 = 그 그룹 총 검출 중 이 품목의 몫

In [1]:
import re
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 60)

df = pd.read_parquet(r"C:\cheon\cheon_wokespace\listeria2026\리스테리아ML\data\listeria_body.parquet")
df["pos"] = (df["judge_result"] == "검출").astype(int)
df["is_mush"] = df["prduct_nm"].astype(str).str.contains("버섯|MUSHROOM|Flammulina|ENOKI|팽이", case=False, regex=True)
print(f"전체 {len(df):,}행 / 검출 {df['pos'].sum():,}건 / 검출률 {df['pos'].mean() * 100:.3f}%")

전체 676,537행 / 검출 1,969건 / 검출률 0.291%


## 11-1. 품목명별 행수·검출·검출률·검출비중 — 전체 / 버섯 제외

In [2]:
def nm_tbl(x):
    t = x.groupby(x["train_prdlst_nm"].astype(str)).agg(행수=("pos", "size"), 검출=("pos", "sum"))
    t["검출률(%)"] = (t["검출"] / t["행수"] * 100).round(3)
    t["행비중(%)"] = (t["행수"] / t["행수"].sum() * 100).round(2)
    t["검출비중(%)"] = (t["검출"] / t["검출"].sum() * 100).round(2)
    return t


nm = pd.concat({"전체": nm_tbl(df), "버섯제외": nm_tbl(df[~df["is_mush"]])}, axis=1)
nm.insert(0, ("", "버섯행"), df.groupby(df["train_prdlst_nm"].astype(str))["is_mush"].sum())
nm = nm.sort_values(("전체", "검출"), ascending=False)
nm.index.name = "train_prdlst_nm"
print(f"품목명 {len(nm)}개 / 검출 1건 이상 {(nm[('전체', '검출')] > 0).sum()}개")
display(nm)

품목명 53개 / 검출 1건 이상 15개


전체                                버섯제외                           
                  버섯행      행수   검출  검출률(%) 행비중(%) 검출비중(%)      행수   검출 검출률(%) 행비중(%) 검출비중(%)
train_prdlst_nm                                                                             
기타기준규격외          1866  291813  791   0.271  43.13   40.17  289947  770  0.266  43.49   53.36
정액,수정란,종란        7433   28651  640   2.234   4.23   32.50   21218  147  0.693   3.18   10.19
식육가공품 및 포장육        79  101045  323   0.320  14.94   16.40  100966  323  0.320  15.14   22.38
수산물                 0    8473   46   0.543   1.25    2.34    8473   46  0.543   1.27    3.19
즉석식품류             177   35322   45   0.127   5.22    2.29   35145   45  0.128   5.27    3.12
유가공품류              16  107022   40   0.037  15.82    2.03  107006   40  0.037  16.05    2.77
수산가공식품류             0    4915   29   0.590   0.73    1.47    4915   29  0.590   0.74    2.01
축산물                 0    1731   28   1.618   0.26    1.42    1731   28  1.618   0.26    1.94
버섯류                83      92   12  13.043   0.01    0.61       9    0  0.000   0.00    0.00
기타식품류              32    7384    9   0.122   1.09    0.46    7352    9  0.122   1.10    0.62
기타식물류               0      43    2   4.651   0.01    0.10      43    2  4.651   0.01    0.14
음료류                11    6398    1   0.016   0.95    0.05    6387    1  0.016   0.96    0.07
견과 종실류              0     292    1   0.342   0.04    0.05     292    1  0.342   0.04    0.07
과자류, 빵류 또는 떡류       0    3936    1   0.025   0.58    0.05    3936    1  0.025   0.59    0.07
농산가공식품류            12    2923    1   0.034   0.43    0.05    2911    1  0.034   0.44    0.07
채소류                 0     904    0   0.000   0.13    0.00     904    0  0.000   0.14    0.00
절임류 또는 조림류          3    2804    0   0.000   0.41    0.00    2801    0  0.000   0.42    0.00
특수영양식품              1    2140    0   0.000   0.32    0.00    2139    0  0.000   0.32    0.00
혼합제제류               0      68    0   0.000   0.01    0.00      68    0  0.000   0.01    0.00
인삼.홍삼제품류            0       3    0   0.000   0.00    0.00       3    0  0.000   0.00    0.00
장류                  0     430    0   0.000   0.06    0.00     430    0  0.000   0.06    0.00
코코아가공품류 또는 초콜릿류     0     735    0   0.000   0.11    0.00     735    0  0.000   0.11    0.00
잼류                  0     148    0   0.000   0.02    0.00     148    0  0.000   0.02    0.00
접객용 음용수             0      50    0   0.000   0.01    0.00      50    0  0.000   0.01    0.00
영양성분                0      12    0   0.000   0.00    0.00      12    0  0.000   0.00    0.00
환경검체                0      35    0   0.000   0.01    0.00      35    0  0.000   0.01    0.00
조류                  0       1    0   0.000   0.00    0.00       1    0  0.000   0.00    0.00
특수의료용도식품            1      82    0   0.000   0.01    0.00      81    0  0.000   0.01    0.00
조미식품                6    4165    0   0.000   0.62    0.00    4159    0  0.000   0.62    0.00
주류                  0      74    0   0.000   0.01    0.00      74    0  0.000   0.01    0.00
조리식품 등            102    9590    0   0.000   1.42    0.00    9488    0  0.000   1.42    0.00
가공식품(전부개정)          0       3    0   0.000   0.00    0.00       3    0  0.000   0.00    0.00
알가공품류              20   29392    0   0.000   4.34    0.00   29372    0  0.000   4.41    0.00
동물성가공식품류            0     189    0   0.000   0.03    0.00     189    0  0.000   0.03    0.00
곡류                  0      52    0   0.000   0.01    0.00      52    0  0.000   0.01    0.00
과일류                 0     702    0   0.000   0.10    0.00     702    0  0.000   0.11    0.00
기능성 원료              0     271    0   0.000   0.04    0.00     271    0  0.000   0.04    0.00
기타                  0     170    0   0.000   0.03    0.00     170    0  0.000   0.03    0.00
기타제조가공식품류(자가규격)     0     342    0   0.000   0.05    0.00     342    0  0.000   0.05    0.00
기타첨가물               0       1    0   0.000   0.00    0.00       1    0  0.000   0.00    0.00
농산물 종자류             0       

## 11-2. 검출률 기준 정렬 (행수 ≥ 1,000) — 평균보다 높은 품목

In [3]:
base = df["pos"].mean() * 100
big = nm[nm[("전체", "행수")] >= 1000].copy()
big[("전체", "평균대비(배)")] = (big[("전체", "검출률(%)")] / base).round(2)
print(f"전체 평균 검출률 {base:.3f}% / 행수 ≥ 1,000 품목 {len(big)}개")
display(big.sort_values(("전체", "검출률(%)"), ascending=False))

전체 평균 검출률 0.291% / 행수 ≥ 1,000 품목 18개


전체                               버섯제외                                 전체
                  버섯행      행수   검출 검출률(%) 행비중(%) 검출비중(%)      행수   검출 검출률(%) 행비중(%) 검출비중(%) 평균대비(배)
train_prdlst_nm                                                                                    
정액,수정란,종란        7433   28651  640  2.234   4.23   32.50   21218  147  0.693   3.18   10.19    7.68
축산물                 0    1731   28  1.618   0.26    1.42    1731   28  1.618   0.26    1.94    5.56
수산가공식품류             0    4915   29  0.590   0.73    1.47    4915   29  0.590   0.74    2.01    2.03
수산물                 0    8473   46  0.543   1.25    2.34    8473   46  0.543   1.27    3.19    1.87
식육가공품 및 포장육        79  101045  323  0.320  14.94   16.40  100966  323  0.320  15.14   22.38    1.10
기타기준규격외          1866  291813  791  0.271  43.13   40.17  289947  770  0.266  43.49   53.36    0.93
즉석식품류             177   35322   45  0.127   5.22    2.29   35145   45  0.128   5.27    3.12    0.44
기타식품류              32    7384    9  0.122   1.09    0.46    7352    9  0.122   1.10    0.62    0.42
유가공품류              16  107022   40  0.037  15.82    2.03  107006   40  0.037  16.05    2.77    0.13
농산가공식품류            12    2923    1  0.034   0.43    0.05    2911    1  0.034   0.44    0.07    0.12
과자류, 빵류 또는 떡류       0    3936    1  0.025   0.58    0.05    3936    1  0.025   0.59    0.07    0.09
음료류                11    6398    1  0.016   0.95    0.05    6387    1  0.016   0.96    0.07    0.05
절임류 또는 조림류          3    2804    0  0.000   0.41    0.00    2801    0  0.000   0.42    0.00    0.00
특수영양식품              1    2140    0  0.000   0.32    0.00    2139    0  0.000   0.32    0.00    0.00
조미식품                6    4165    0  0.000   0.62    0.00    4159    0  0.000   0.62    0.00    0.00
조리식품 등            102    9590    0  0.000   1.42    0.00    9488    0  0.000   1.42    0.00    0.00
알가공품류              20   29392    0  0.000   4.34    0.00   29372    0  0.000   4.41    0.00    0.00
빙과류                 9   22206    0  0.000   3.28    0.00   22197    0  0.000   3.33    0.00    0.00

## 11-3. 검출 0건 품목

In [4]:
z = nm[nm[("전체", "검출")] == 0][[("전체", "행수")]]
print(f"검출 0건 품목 {len(z)}개 / 행 {int(z.sum().iloc[0]):,}행")
display(z.sort_values(("전체", "행수"), ascending=False))

검출 0건 품목 38개 / 행 76,497행


,전체
,행수
train_prdlst_nm,
알가공품류,29392
빙과류,22206
조리식품 등,9590
조미식품,4165
절임류 또는 조림류,2804
특수영양식품,2140
채소류,904
코코아가공품류 또는 초콜릿류,735


## 정리 (셀 출력 수치만)

- 품목명 53개 중 검출 1건 이상 15개, 검출 0건 38개(76,497행)
- 검출 상위: 기타기준규격외 791건(0.271%, 검출비중 40.17%), 정액,수정란,종란 640건(2.234%, 32.50%), 식육가공품 및 포장육 323건(0.320%)
- 정액,수정란,종란 은 버섯 행 7,433개를 빼면 검출률 2.234% → 0.693% (그래도 평균의 2배 이상)
- 행수 ≥ 1,000 품목 중 검출률 2위는 축산물 1.618% (1,731행, 28건, 버섯행 0)
- 알가공품류 29,392행, 빙과류 22,206행 은 검출 0건